# Polyp Detection — Error Analysis

Before choosing a fix for the CVC-ClinicDB recall gap, this notebook
identifies exactly which images the model misses and looks for
patterns that explain why — rather than guessing at a solution.

The analysis tests two independent hypotheses: polyp size (are missed
polyps systematically smaller?) and polyp position (do misses cluster
near frame edges?).

---

## What this notebook does

1. Runs the baseline model on every CVC-ClinicDB image (conf=0.10,
   to find the absolute ceiling of what the model can detect)
2. Identifies missed polyps using IoU matching (threshold=0.3)
3. Compares missed vs. detected polyps on size and position
4. Visualizes the smallest missed cases with ground truth overlay

## Output

- results/figures/error_analysis_missed_samples.png
- results/figures/error_analysis_size_comparison.png
- results/figures/error_analysis_position.png
- results/metrics/error_analysis_summary.json

In [ ]:
# Import libraries

import json
import os
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from ultralytics import YOLO

In [ ]:
# Config & Paths
# Project paths and settings

BASE_DIR = r"D:\Deep_Projects\polyp-detection-safety-first\repo"

PATHS = {
    "baseline_weights": os.path.join(BASE_DIR, "models", "baseline", "weights", "best.pt"),
    "cvc_images":       os.path.join(BASE_DIR, "data", "yolo_format", "cvc_test", "images"),
    "cvc_labels":       os.path.join(BASE_DIR, "data", "yolo_format", "cvc_test", "labels"),
    "figures":          os.path.join(BASE_DIR, "results", "figures"),
    "metrics":          os.path.join(BASE_DIR, "results", "metrics"),
}

os.makedirs(PATHS["figures"], exist_ok=True)
os.makedirs(PATHS["metrics"], exist_ok=True)

print("Paths configured:")
for name, path in PATHS.items():
    status = "OK" if os.path.exists(path) else "missing"
    print(f"  [{status}] {name:16s} -> {path}")

In [ ]:
# Load Model

print(f"Loading baseline model: {PATHS['baseline_weights']}")
model = YOLO(PATHS["baseline_weights"])
print("Model loaded successfully.")

In [ ]:
# Ground Truth Loader
# Read YOLO-seg polygon labels and convert to bounding boxes
# (we use boxes here since IoU-based matching is simpler with boxes,
# and this is purely diagnostic - not the final evaluation metric)

def read_yolo_label_as_boxes(label_path, img_width, img_height):
    boxes = []
    if not os.path.exists(label_path):
        return boxes

    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            coords = list(map(float, parts[1:]))
            points = np.array(coords).reshape(-1, 2)
            points[:, 0] *= img_width
            points[:, 1] *= img_height

            x_min, y_min = points.min(axis=0)
            x_max, y_max = points.max(axis=0)
            boxes.append([x_min, y_min, x_max, y_max])

    return boxes


def box_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_area = max(0, x2 - x1) * max(0, y2 - y1)
    if inter_area == 0:
        return 0.0

    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = area1 + area2 - inter_area

    return inter_area / union_area

In [ ]:
# Find Missed Detections
# For every CVC-ClinicDB image, compare ground-truth boxes against
# model predictions. A ground-truth box with no matching prediction
# (IoU < threshold) counts as a missed detection (false negative).

IOU_MATCH_THRESHOLD = 0.3  # lenient on purpose - we just want to know
                            # if the model found the polyp at all,
                            # not whether the box was pixel-perfect

image_files = sorted(os.listdir(PATHS["cvc_images"]))

missed_records   = []
detected_records = []

for fname in image_files:
    stem = os.path.splitext(fname)[0]
    img_path   = os.path.join(PATHS["cvc_images"], fname)
    label_path = os.path.join(PATHS["cvc_labels"], stem + ".txt")

    img = Image.open(img_path)
    w, h = img.size

    gt_boxes = read_yolo_label_as_boxes(label_path, w, h)
    if not gt_boxes:
        continue

    pred = model.predict(img_path, conf=0.10, verbose=False)[0]
    pred_boxes = pred.boxes.xyxy.cpu().numpy().tolist() if len(pred.boxes) > 0 else []

    for gt_box in gt_boxes:
        gt_area = (gt_box[2] - gt_box[0]) * (gt_box[3] - gt_box[1])
        gt_area_ratio = gt_area / (w * h)
        gt_center_x = (gt_box[0] + gt_box[2]) / 2 / w
        gt_center_y = (gt_box[1] + gt_box[3]) / 2 / h

        best_iou = max(
            (box_iou(gt_box, pb) for pb in pred_boxes), default=0.0
        )

        record = {
            "image": fname,
            "area_ratio": gt_area_ratio,
            "center_x": gt_center_x,
            "center_y": gt_center_y,
            "best_iou": best_iou,
        }

        if best_iou < IOU_MATCH_THRESHOLD:
            missed_records.append(record)
        else:
            detected_records.append(record)

print(f"Total ground-truth polyps: {len(missed_records) + len(detected_records)}")
print(f"Detected (IoU >= {IOU_MATCH_THRESHOLD}): {len(detected_records)}")
print(f"Missed:                      {len(missed_records)}")
print(f"Miss rate: {len(missed_records) / (len(missed_records) + len(detected_records)) * 100:.1f}%")

In [ ]:
# Compare Size: Missed vs Detected
# Core question: are missed polyps systematically smaller than
# detected ones? If yes, this points to a small-object detection
# problem rather than a brightness/angle problem.

missed_areas   = [r["area_ratio"] for r in missed_records]
detected_areas = [r["area_ratio"] for r in detected_records]

print("POLYP AREA COMPARISON")
print("-" * 40)
print(f"Detected polyps - median area: {np.median(detected_areas)*100:.2f}%")
print(f"Missed polyps   - median area: {np.median(missed_areas)*100:.2f}%")
print(f"Detected polyps - mean area:   {np.mean(detected_areas)*100:.2f}%")
print(f"Missed polyps   - mean area:   {np.mean(missed_areas)*100:.2f}%")

fig, ax = plt.subplots(figsize=(7, 5))
ax.boxplot(
    [detected_areas, missed_areas],
    tick_labels=["Detected", "Missed"],
)
ax.set_ylabel("Polyp area (fraction of image)")
ax.set_title("Polyp Size: Detected vs Missed (CVC-ClinicDB)")
ax.grid(alpha=0.3)

plt.tight_layout()
save_path = os.path.join(PATHS["figures"], "error_analysis_size_comparison.png")
plt.savefig(save_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")

In [ ]:
# Compare Position: Missed vs Detected
# Secondary question: are missed polyps concentrated near the
# image edges (where the endoscope view is often darker or
# distorted)?

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

detected_x = [r["center_x"] for r in detected_records]
detected_y = [r["center_y"] for r in detected_records]
missed_x   = [r["center_x"] for r in missed_records]
missed_y   = [r["center_y"] for r in missed_records]

axes[0].scatter(detected_x, detected_y, alpha=0.4, s=20,
                 color="steelblue", label="Detected")
axes[0].scatter(missed_x, missed_y, alpha=0.7, s=30,
                 color="red", label="Missed")
axes[0].set_xlim(0, 1)
axes[0].set_ylim(1, 0)
axes[0].set_xlabel("Normalized X position")
axes[0].set_ylabel("Normalized Y position")
axes[0].set_title("Polyp Position in Frame")
axes[0].legend()
axes[0].set_aspect("equal")

axes[1].hist(detected_areas, bins=20, alpha=0.6, color="steelblue",
             label="Detected", density=True)
axes[1].hist(missed_areas, bins=20, alpha=0.6, color="red",
             label="Missed", density=True)
axes[1].set_xlabel("Polyp area (fraction of image)")
axes[1].set_ylabel("Density")
axes[1].set_title("Size Distribution")
axes[1].legend()

plt.tight_layout()
save_path = os.path.join(PATHS["figures"], "error_analysis_position.png")
plt.savefig(save_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")

In [ ]:
# Visualize Sample Missed Cases
# Show actual missed images with ground truth box overlaid,
# sorted by size (smallest first)

missed_sorted = sorted(missed_records, key=lambda r: r["area_ratio"])
sample_missed = missed_sorted[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for ax, record in zip(axes, sample_missed):
    img_path = os.path.join(PATHS["cvc_images"], record["image"])
    img = np.array(Image.open(img_path).convert("RGB"))
    h, w = img.shape[:2]

    stem = os.path.splitext(record["image"])[0]
    label_path = os.path.join(PATHS["cvc_labels"], stem + ".txt")
    gt_boxes = read_yolo_label_as_boxes(label_path, w, h)

    ax.imshow(img)
    for box in gt_boxes:
        x1, y1, x2, y2 = box
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                              fill=False, edgecolor="red", linewidth=2)
        ax.add_patch(rect)

    ax.set_title(f"{record['image']}\narea={record['area_ratio']*100:.1f}%",
                 fontsize=9)
    ax.axis("off")

plt.suptitle("Smallest Missed Polyps (red box = ground truth)",
              fontweight="bold")
plt.tight_layout()
save_path = os.path.join(PATHS["figures"], "error_analysis_missed_samples.png")
plt.savefig(save_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")

In [ ]:
# Save Summary

summary = {
    "total_polyps": len(missed_records) + len(detected_records),
    "missed_count": len(missed_records),
    "detected_count": len(detected_records),
    "miss_rate": len(missed_records) / (len(missed_records) + len(detected_records)),
    "median_area_detected": float(np.median(detected_areas)),
    "median_area_missed": float(np.median(missed_areas)),
    "mean_area_detected": float(np.mean(detected_areas)),
    "mean_area_missed": float(np.mean(missed_areas)),
}

summary_path = os.path.join(PATHS["metrics"], "error_analysis_summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved -> {summary_path}")
print()
print(json.dumps(summary, indent=2))

In [ ]:
# Summary

print("NOTEBOOK 04b COMPLETE")
print("=" * 50)
size_gap = (summary["median_area_detected"] - summary["median_area_missed"]) / summary["median_area_detected"] * 100
print(f"Missed polyps are {size_gap:.0f}% smaller (median) than detected ones")
print()
print("Interpretation guide:")
print("  - If missed polyps are much smaller -> small-object detection")
print("    problem. Consider: higher imgsz, zoomed training crops.")
print("  - If missed polyps are similar size but scattered randomly")
print("    -> brightness/texture problem. TTA may help.")
print("  - If missed polyps cluster near edges -> endoscope view")
print("    distortion. Consider: edge-aware augmentation.")